In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "./modules/python-utils",
    "./modules/ai-utils",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path
import librosa
import time

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.libri_speech_asr_corpus import search_all_ref_and_hyp
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.audio_utils import *
from sj_utils.string_utils import *
from sj_utils.collection_utils import SafetyDict

In [ ]:
from rt_whisper import streamers
from rt_whisper.data import Param, Result

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/test-other/LibriSpeech/test-other/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000

In [ ]:
hyperparameters = SafetyDict({
    "sentence_max_prev_sentence": 1,
    "weighted_and_offset_token_boundary": 8000,
    "duration_filter_z":{
        "default": 2.0,
        "ko": 2.0,
        "en": 3.0,
    },
    "probability_filter":{
        "z":{
            "default": 3.0,
            "ko": 3.0,
            "en": 3.0,
        },
        "min_prob": {
            "default": 1.0,
            "ko": 0.4,
            "en": 0.4
        },
    },
    "selector":{
        "search_range_sc": {
            "default": 24000,
            "ko": 24000,
            "en": 20000,
        },
        "threshold":{
            "default": 0.5,
            "ko": 0.25,
            "en": 0.25
        },
        "padding": {
            "default": 3200,
            "ko": 3200,
            "en": 3200
        },
        "tolerance": {
            "default": 8000,
            "ko": 8000,
            "en": 8000
        }
    },
    "max_overlap_duration": 96000
})

In [ ]:
token_streamer = streamers.get_token_streamer_with_vad_v2(hyperparameter= hyperparameters)

In [ ]:
src = Path(SOURCE)

In [ ]:
def transcriber(flac:Path) -> TRNFormat:
    audio, _ = librosa.load(flac, sr=SAMPLE_RATE)

    completed = []
    param = Param()

    print(f"Processing")
    print(f"\tAudio name: {flac.name}")
    print(f"\tAudio length: {len(audio) / SAMPLE_RATE:.2f} seconds")
    start_time = time.perf_counter()
    for segment in segment_audio(audio):
        param.chunk = segment
        param.language="en"
        result:Result = token_streamer.process(param)
        completed.extend(result.completed)
        param.update(result)
    completed.extend(result.candidate)
    end_time = time.perf_counter()
    print(f"\tProcessed time: {end_time - start_time:.2f} seconds")

    return TRNFormat(
        id = flac.stem,
        text = normalize_text_only_en(
            " ".join([s.text for s in completed])
        ).upper()
    )

In [ ]:
%%time
data = search_all_ref_and_hyp(src, transcriber, 5)
# CPU times: user 10min 9s, sys: 14.9 s, total: 10min 24s
# Wall time: 1min 41s

In [ ]:
concat_result = {}
for value in data.values():
    for k, v in value.items():
        if k not in concat_result:
            concat_result[k] = []
        concat_result[k].extend(v)

In [ ]:
output = sclite_trn(
    concat_result["ref"],
    concat_result["hyp"],
)

In [ ]:
parse_sclite_summary(output)

# {'num_sentences': 96,
#  'num_words': 1472,
#  'correct_percent': 92.5,
#  'substitution_percent': 6.2,
#  'deletion_percent': 1.4,
#  'insertion_percent': 0.3,
#  'wer_percent': 7.9,
#  'sentence_error_percent': 61.5}